In [ ]:
%load_ext autoreload
%autoreload 2
import os

os.chdir("..")
print(os.getcwd())

In [ ]:
import pandas as pd
import requests
import wandb

In [ ]:
def pprint(dictionary):
    for k, v in dictionary.items():
        print(k, ":", v)

In [ ]:
def get_latex(df, columns=None):
    if columns is not None:
        df = df[columns]
    else:
        columns = list(df.columns)
    latex_table = df.to_latex(float_format="%.3f", escape=True, index=False, columns=columns)
    print(latex_table)

    payload = {
        "formula": latex_table,
        "fsize": "54px",
        "fcolor": "000000",
        "mode": "0",  # 0 = LaTeX inline/display mode won't handle tabular well; see note below
        "out": "1",
        "remhost": "quicklatex.com",
        "preamble": r"\usepackage{booktabs}\usepackage{amsmath}",
    }

    response = requests.post("https://quicklatex.com/latex3.f", data=payload)
    print(response.text)

In [ ]:
os.makedirs("outputs/prediction", exist_ok=True)

In [ ]:
results_df_path = "outputs/prediction/results.csv"
os.makedirs("outputs/prediction/results", exist_ok=True)
if os.path.exists(results_df_path):
    df = pd.read_csv(results_df_path)
    print(f"There are {len(df)} reccords already in results.")
    df = df.loc[:, ~df.columns.str.contains("^Unnamed")]
else:
    df = None

In [ ]:
# Connect to wandb api
api = wandb.Api()
entity = "aether_xai"
project = "s2bms_prediction"

In [ ]:
runs_iterator = api.runs(f"{entity}/{project}")
run_list = []

for i, run in enumerate(runs_iterator):
    if df is not None:
        if run.id in list(df.run_id):
            print(
                f'{run.summary["experiment"]} with seed={run.config["seed"]} already in results.'
            )
            continue
    elif run.state != "finished":
        print(f"{run.id} is not finished.")
        continue

    captures = dict(run.summary)
    captures["run_id"] = run.id
    captures["seed"] = run.config["seed"]
    run_list.append(captures)
    print(f'{run.summary["experiment"]} with seed={run.config["seed"]} logged.')

In [ ]:
runs_df = pd.DataFrame(run_list)
if df is not None:
    runs_df = pd.concat([df, runs_df], ignore_index=True)
runs_df.to_csv("outputs/prediction/results.csv", index=False)

In [ ]:
runs_df.columns

In [ ]:
cols_of_interest = {
    "experiment": "Modality (best config.)",
    "best_val_loss": "Validation loss",
    "test_mse_loss": "Test MSE loss",
    "test_top_1_acc": "Test Top-1 acc",
    "test_top_5_acc": "Test Top-5 acc",
    "train_mse_loss": "Train MSE loss",
    "test_top_10_acc": "Test Top-10 acc",
    "best_val_mse_loss": "Validation MSE loss",
    "best_val_top_1_acc": "Validation Top-1 acc",
    "best_val_top_5_acc": "Validation Top-5 acc",
    "best_val_top_10_acc": "Validation Top-10 acc",
}


sub_runs_df = runs_df[cols_of_interest.keys()]
for col in cols_of_interest.keys():
    if "top" in col:
        sub_runs_df[col] = sub_runs_df[col].apply(lambda x: x * 100)
summary = sub_runs_df.groupby("experiment").mean(numeric_only=True)
summary.sort_values("test_mse_loss")
summary.reset_index(inplace=True)

In [ ]:
modalities = {
    "aef": "AlphaEarth",
    "tessera": "Tessera",
    "s2": "Sentinel-2",
    "baseline_mlp": "Baseline-MLP",
    "baseline_lin": "Baseline-Lin",
    "geoclip": "GeoCLIP",
    "satclip": "SatCLIP",
}


def parse_modality(x):
    for k, v in modalities.items():
        if k in x:
            return v

In [ ]:
summary["modality"] = summary["experiment"].apply(lambda x: parse_modality(x))
cols_of_interest["modality"] = "Modality (best config.)"

In [ ]:
summary

# Table 1

In [ ]:
# selection_col = 'test_top_5_acc'
selection_col = "test_top_10_acc"
# selection_col = 'test_mse_loss'
# selection_col = 'best_val_mse_loss'
selection_mode = "max"
# selection_mode = 'min'


if selection_mode == "max":
    tab_1 = summary.loc[summary.groupby("modality")[selection_col].idxmax()]
    tab_1 = tab_1.sort_values(selection_col, ascending=False)  # best (highest) first
else:
    tab_1 = summary.loc[summary.groupby("modality")[selection_col].idxmin()]
    tab_1 = tab_1.sort_values(selection_col, ascending=True)  # best (lowest) first

tab_1[["modality", "experiment", selection_col]]

In [ ]:
keep = [
    "modality",
    "best_val_loss",
    "test_mse_loss",
    "test_top_1_acc",
    "test_top_5_acc",
    "train_mse_loss",
    "test_top_10_acc",
    "best_val_mse_loss",
    "best_val_top_1_acc",
    "best_val_top_5_acc",
    "best_val_top_10_acc",
]

# keep = ['experiment', 'best_val_loss', 'test_mse_loss', 'test_top_1_acc', 'test_top_5_acc', 'train_mse_loss', 'test_top_10_acc', 'best_val_mse_loss', 'best_val_top_1_acc', 'best_val_top_5_acc', 'best_val_top_10_acc', 'modality']

tab_1 = tab_1[keep]
tab_1.rename(columns=cols_of_interest, inplace=True)
tab_1

In [ ]:
get_latex(
    tab_1,
    columns=["modality", "test_top_10_acc", "test_top_5_acc", "test_top_1_acc", "test_mse_loss"],
)

# Table S1 per modality

In [ ]:
for m in modalities.values():
    tab_s1 = summary[summary["modality"] == m]
    if selection_mode == "max":
        tab_s1.sort_values(selection_col, ascending=False, inplace=True)
    else:
        tab_s1.sort_values(selection_col, ascending=True, inplace=True)

    tab_s1

In [ ]:
tab_s1 = summary[summary["modality"] == "AlphaEarth"]
if selection_mode == "max":
    tab_s1.sort_values(selection_col, ascending=False, inplace=True)
else:
    tab_s1.sort_values(selection_col, ascending=True, inplace=True)
# keep = ['modality', 'best_val_loss', 'test_mse_loss', 'test_top_1_acc', 'test_top_5_acc', 'train_mse_loss',  'best_val_mse_loss', 'best_val_top_1_acc', 'best_val_top_5_acc', 'best_val_top_10_acc']
get_latex(
    tab_s1,
    columns=["experiment", "test_top_10_acc", "test_top_5_acc", "test_top_1_acc", "test_mse_loss"],
)
tab_s1

In [ ]:
summary